# Fabric Tenant Settings Change Tracker

Tracks **Fabric tenant settings** changes using **Spark SQL**:
1. Fetches current tenant settings via Fabric Admin API
2. Stores daily snapshots in a Delta table (MERGE INTO)
3. Compares yesterday vs today using SQL EXCEPT / JOIN to find:
   - **New** settings | **Removed** settings | **Changed** settings
4. Exits with JSON payload for the wrapping pipeline to send Outlook email

> **Prerequisites**: Attach a lakehouse. Fabric Admin privileges required.
> Designed to run from **Tenant_Settings_Alert_Pipeline**.

In [ ]:
# Cell 1: Configuration
# Use fully-qualified lakehouse names so the notebook works with or without
# a default lakehouse attached (e.g. when invoked from a pipeline via API deployment).
SNAPSHOT_TABLE = "TenantSettings.tenant_settings_snapshots"
CHANGES_TABLE  = "TenantSettings.tenant_settings_changes"


In [ ]:
# Cell 2: Fetch tenant settings from Fabric Admin API
import requests, json as _json
from datetime import datetime, timedelta
from notebookutils import mssparkutils

token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
response = requests.get(
    "https://api.fabric.microsoft.com/v1/admin/tenantsettings",
    headers={"Authorization": f"Bearer {token}"}
)
response.raise_for_status()
raw_settings = response.json().get("tenantSettings", [])
print(f"Fetched {len(raw_settings)} tenant settings")

In [ ]:
# Cell 3: Flatten API response into a temp view
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
today = str(datetime.utcnow().date())
yesterday = str(datetime.utcnow().date() - timedelta(days=1))

rows = []
for s in raw_settings:
    rows.append((
        today,
        s.get("settingName", ""),
        s.get("title", ""),
        s.get("enabled"),
        s.get("canSpecifySecurityGroups"),
        s.get("tenantSettingGroup", ""),
        _json.dumps(s.get("enabledSecurityGroups", []), sort_keys=True),
        _json.dumps(s.get("properties", []), sort_keys=True),
        s.get("delegatedFrom", ""),
        s.get("delegateState", ""),
    ))

columns = [
    "snapshot_date", "setting_name", "title", "enabled",
    "can_specify_security_groups", "tenant_setting_group",
    "enabled_security_groups", "properties",
    "delegated_from", "delegated_state"
]

df_today = spark.createDataFrame(rows, columns)
df_today.createOrReplaceTempView("today_settings")

spark.sql("SELECT count(*) AS cnt FROM today_settings").show()
spark.sql("SELECT setting_name, title, enabled FROM today_settings LIMIT 5").show(truncate=40)

In [ ]:
# Cell 4: Upsert today's snapshot into Delta table using MERGE INTO

if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    spark.sql(f"""
        CREATE TABLE {SNAPSHOT_TABLE}
        USING DELTA
        AS SELECT * FROM today_settings
    """)
    print(f"Created '{SNAPSHOT_TABLE}' with today's snapshot")
else:
    spark.sql(f"""
        MERGE INTO {SNAPSHOT_TABLE} AS target
        USING today_settings AS source
        ON target.snapshot_date = source.snapshot_date
           AND target.setting_name = source.setting_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged today's snapshot into '{SNAPSHOT_TABLE}'")

In [ ]:
# Cell 5: Load previous (last available) + today from Delta table into views

# Dynamically find the most recent snapshot before today — works for any run frequency
# (daily, weekly, ad-hoc). Avoids missing changes when a day is skipped.
prev_row = spark.sql(f"""
    SELECT MAX(snapshot_date) AS prev_date
    FROM {SNAPSHOT_TABLE}
    WHERE snapshot_date < '{today}'
""").collect()[0]
prev_date = str(prev_row["prev_date"]) if prev_row["prev_date"] else None

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW v_yesterday AS
    SELECT * FROM {SNAPSHOT_TABLE}
    WHERE snapshot_date = '{prev_date or "1900-01-01"}'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW v_today AS
    SELECT * FROM {SNAPSHOT_TABLE}
    WHERE snapshot_date = '{today}'
""")

yesterday_count = spark.sql("SELECT count(*) AS cnt FROM v_yesterday").collect()[0]["cnt"]
today_count = spark.sql("SELECT count(*) AS cnt FROM v_today").collect()[0]["cnt"]

print(f"Previous ({prev_date or 'none'}): {yesterday_count} settings")
print(f"Today     ({today}): {today_count} settings")

if yesterday_count == 0:
    print(f"\nNo previous snapshot found before {today}. Baseline saved — run again after the next change.")


In [ ]:
# Cell 6: Find NEW and REMOVED settings using SQL EXCEPT

new_count = 0
removed_count = 0

if yesterday_count > 0:
    # NEW = setting_name in today EXCEPT yesterday
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW v_new_settings AS
        SELECT t.*
        FROM v_today t
        WHERE t.setting_name IN (
            SELECT setting_name FROM v_today
            EXCEPT
            SELECT setting_name FROM v_yesterday
        )
    """)
    new_count = spark.sql("SELECT count(*) AS cnt FROM v_new_settings").collect()[0]["cnt"]

    print(f"\n{'='*60}")
    print(f"  NEW SETTINGS (appeared today): {new_count}")
    print(f"{'='*60}")
    if new_count > 0:
        spark.sql("SELECT setting_name, title, enabled, tenant_setting_group FROM v_new_settings").show(100, truncate=60)
    else:
        print("  No new settings found.")

    # REMOVED = setting_name in yesterday EXCEPT today
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW v_removed_settings AS
        SELECT y.*
        FROM v_yesterday y
        WHERE y.setting_name IN (
            SELECT setting_name FROM v_yesterday
            EXCEPT
            SELECT setting_name FROM v_today
        )
    """)
    removed_count = spark.sql("SELECT count(*) AS cnt FROM v_removed_settings").collect()[0]["cnt"]

    print(f"\n{'='*60}")
    print(f"  REMOVED SETTINGS (gone today): {removed_count}")
    print(f"{'='*60}")
    if removed_count > 0:
        spark.sql("SELECT setting_name, title, enabled, tenant_setting_group FROM v_removed_settings").show(100, truncate=60)
    else:
        print("  No settings were removed.")
else:
    print("Skipping -- no yesterday snapshot.")

In [ ]:
# Cell 7: Find CHANGED settings using SQL JOIN + WHERE

changed_count = 0

if yesterday_count > 0:
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW v_changed_settings AS
        SELECT
            y.setting_name,
            y.title,
            y.enabled               AS y_enabled,
            t.enabled               AS t_enabled,
            y.can_specify_security_groups AS y_can_specify_sg,
            t.can_specify_security_groups AS t_can_specify_sg,
            y.tenant_setting_group  AS y_group,
            t.tenant_setting_group  AS t_group,
            y.enabled_security_groups AS y_sec_groups,
            t.enabled_security_groups AS t_sec_groups,
            y.properties            AS y_properties,
            t.properties            AS t_properties,
            y.delegated_from        AS y_delegated_from,
            t.delegated_from        AS t_delegated_from,
            y.delegated_state       AS y_delegated_state,
            t.delegated_state       AS t_delegated_state
        FROM v_yesterday y
        INNER JOIN v_today t ON y.setting_name = t.setting_name
        WHERE
            y.enabled <> t.enabled
            OR (y.enabled IS NULL AND t.enabled IS NOT NULL)
            OR (y.enabled IS NOT NULL AND t.enabled IS NULL)
            OR y.can_specify_security_groups <> t.can_specify_security_groups
            OR (y.can_specify_security_groups IS NULL AND t.can_specify_security_groups IS NOT NULL)
            OR (y.can_specify_security_groups IS NOT NULL AND t.can_specify_security_groups IS NULL)
            OR y.tenant_setting_group <> t.tenant_setting_group
            OR y.enabled_security_groups <> t.enabled_security_groups
            OR y.properties <> t.properties
            OR y.delegated_from <> t.delegated_from
            OR y.delegated_state <> t.delegated_state
    """)

    changed_count = spark.sql("SELECT count(*) AS cnt FROM v_changed_settings").collect()[0]["cnt"]

    print(f"\n{'='*60}")
    print(f"  CHANGED SETTINGS: {changed_count}")
    print(f"{'='*60}")

    if changed_count > 0:
        # Show readable diff
        compare_pairs = [
            ("y_enabled", "t_enabled", "enabled"),
            ("y_can_specify_sg", "t_can_specify_sg", "can_specify_security_groups"),
            ("y_group", "t_group", "tenant_setting_group"),
            ("y_sec_groups", "t_sec_groups", "enabled_security_groups"),
            ("y_properties", "t_properties", "properties"),
            ("y_delegated_from", "t_delegated_from", "delegated_from"),
            ("y_delegated_state", "t_delegated_state", "delegated_state"),
        ]
        for row in spark.sql("SELECT * FROM v_changed_settings").collect():
            r = row.asDict()
            print(f"\n  Setting: {r['setting_name']}")
            for yc, tc, label in compare_pairs:
                ov, nv = str(r.get(yc, '') or ''), str(r.get(tc, '') or '')
                if ov != nv:
                    print(f"    {label}: '{ov}' -> '{nv}'")
    else:
        print("  No settings changed.")
else:
    print("Skipping -- no yesterday snapshot.")

In [ ]:
# Cell 8: Persist change log to audit Delta table using SQL INSERT

if yesterday_count > 0:
    # Create the changes table if it doesn't exist
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CHANGES_TABLE} (
            change_date    STRING,
            change_type    STRING,
            setting_name   STRING,
            title          STRING,
            field_changed  STRING,
            old_value      STRING,
            new_value      STRING
        ) USING DELTA
    """)

    # Insert NEW settings
    if new_count > 0:
        spark.sql(f"""
            INSERT INTO {CHANGES_TABLE}
            SELECT
                '{today}' AS change_date,
                'NEW'     AS change_type,
                setting_name,
                title,
                ''        AS field_changed,
                ''        AS old_value,
                CAST(enabled AS STRING) AS new_value
            FROM v_new_settings
        """)
        print(f"Logged {new_count} NEW setting(s)")

    # Insert REMOVED settings
    if removed_count > 0:
        spark.sql(f"""
            INSERT INTO {CHANGES_TABLE}
            SELECT
                '{today}'  AS change_date,
                'REMOVED'  AS change_type,
                setting_name,
                title,
                ''         AS field_changed,
                CAST(enabled AS STRING) AS old_value,
                ''         AS new_value
            FROM v_removed_settings
        """)
        print(f"Logged {removed_count} REMOVED setting(s)")

    # Insert CHANGED settings using UNPIVOT-style SQL
    if changed_count > 0:
        spark.sql(f"""
            INSERT INTO {CHANGES_TABLE}
            SELECT
                '{today}'  AS change_date,
                'CHANGED'  AS change_type,
                setting_name,
                title,
                field_name AS field_changed,
                old_val    AS old_value,
                new_val    AS new_value
            FROM (
                SELECT setting_name, title, 'enabled' AS field_name,
                       CAST(y_enabled AS STRING) AS old_val, CAST(t_enabled AS STRING) AS new_val
                FROM v_changed_settings WHERE CAST(y_enabled AS STRING) <> CAST(t_enabled AS STRING)
                   OR (y_enabled IS NULL AND t_enabled IS NOT NULL) OR (y_enabled IS NOT NULL AND t_enabled IS NULL)
                UNION ALL
                SELECT setting_name, title, 'can_specify_security_groups',
                       CAST(y_can_specify_sg AS STRING), CAST(t_can_specify_sg AS STRING)
                FROM v_changed_settings WHERE CAST(y_can_specify_sg AS STRING) <> CAST(t_can_specify_sg AS STRING)
                   OR (y_can_specify_sg IS NULL AND t_can_specify_sg IS NOT NULL) OR (y_can_specify_sg IS NOT NULL AND t_can_specify_sg IS NULL)
                UNION ALL
                SELECT setting_name, title, 'tenant_setting_group',
                       y_group, t_group
                FROM v_changed_settings WHERE y_group <> t_group
                UNION ALL
                SELECT setting_name, title, 'enabled_security_groups',
                       y_sec_groups, t_sec_groups
                FROM v_changed_settings WHERE y_sec_groups <> t_sec_groups
                UNION ALL
                SELECT setting_name, title, 'properties',
                       y_properties, t_properties
                FROM v_changed_settings WHERE y_properties <> t_properties
                UNION ALL
                SELECT setting_name, title, 'delegated_from',
                       y_delegated_from, t_delegated_from
                FROM v_changed_settings WHERE y_delegated_from <> t_delegated_from
                UNION ALL
                SELECT setting_name, title, 'delegated_state',
                       y_delegated_state, t_delegated_state
                FROM v_changed_settings WHERE y_delegated_state <> t_delegated_state
            )
        """)
        logged = spark.sql(f"SELECT count(*) AS cnt FROM {CHANGES_TABLE} WHERE change_date = '{today}' AND change_type = 'CHANGED'").collect()[0]["cnt"]
        print(f"Logged {logged} CHANGED field(s)")

    # Show today's audit log
    spark.sql(f"""
        SELECT change_type, setting_name, field_changed, old_value, new_value
        FROM {CHANGES_TABLE}
        WHERE change_date = '{today}'
        ORDER BY change_type, setting_name
    """).show(100, truncate=50)
else:
    print("First run -- snapshot saved. Run again tomorrow for comparisons.")

In [ ]:
# Cell 9: Build email HTML and exit with JSON for the pipeline Outlook activity

def build_email_html():
    h = []
    h.append("<html><body style='font-family:Segoe UI,Arial,sans-serif;color:#333;'>")
    h.append(f"<h2 style='color:#0078d4;'>Fabric Tenant Settings Change Report</h2>")
    h.append(f"<p><b>Date:</b> {today} | <b>Compared against:</b> {prev_date or 'N/A'}</p><hr/>")
    h.append(f"<table style='border-collapse:collapse;margin-bottom:16px;'>")
    h.append(f"<tr><td style='padding:4px 12px;'>New:</td><td><b>{new_count}</b></td></tr>")
    h.append(f"<tr><td style='padding:4px 12px;'>Removed:</td><td><b>{removed_count}</b></td></tr>")
    h.append(f"<tr><td style='padding:4px 12px;'>Changed:</td><td><b>{changed_count}</b></td></tr></table>")

    if new_count > 0:
        h.append("<h3 style='color:#107c10;'>New Settings</h3>")
        h.append("<table style='border-collapse:collapse;width:100%;border:1px solid #ddd;'>")
        h.append("<tr style='background:#107c10;color:white;'><th style='padding:8px;text-align:left;'>Setting</th>")
        h.append("<th style='padding:8px;text-align:left;'>Title</th><th style='padding:8px;'>Enabled</th><th style='padding:8px;text-align:left;'>Group</th></tr>")
        for r in spark.sql("SELECT setting_name, title, enabled, tenant_setting_group FROM v_new_settings").collect():
            d = r.asDict()
            h.append(f"<tr style='border-bottom:1px solid #ddd;'><td style='padding:6px 8px;'>{d['setting_name']}</td>")
            h.append(f"<td style='padding:6px 8px;'>{d.get('title','')}</td>")
            h.append(f"<td style='padding:6px 8px;text-align:center;'>{d.get('enabled','')}</td>")
            h.append(f"<td style='padding:6px 8px;'>{d.get('tenant_setting_group','')}</td></tr>")
        h.append("</table><br/>")

    if removed_count > 0:
        h.append("<h3 style='color:#d83b01;'>Removed Settings</h3>")
        h.append("<table style='border-collapse:collapse;width:100%;border:1px solid #ddd;'>")
        h.append("<tr style='background:#d83b01;color:white;'><th style='padding:8px;text-align:left;'>Setting</th>")
        h.append("<th style='padding:8px;text-align:left;'>Title</th><th style='padding:8px;'>Was Enabled</th><th style='padding:8px;text-align:left;'>Group</th></tr>")
        for r in spark.sql("SELECT setting_name, title, enabled, tenant_setting_group FROM v_removed_settings").collect():
            d = r.asDict()
            h.append(f"<tr style='border-bottom:1px solid #ddd;'><td style='padding:6px 8px;'>{d['setting_name']}</td>")
            h.append(f"<td style='padding:6px 8px;'>{d.get('title','')}</td>")
            h.append(f"<td style='padding:6px 8px;text-align:center;'>{d.get('enabled','')}</td>")
            h.append(f"<td style='padding:6px 8px;'>{d.get('tenant_setting_group','')}</td></tr>")
        h.append("</table><br/>")

    if changed_count > 0:
        h.append("<h3 style='color:#ca5010;'>Changed Settings (Old &#8594; New)</h3>")
        h.append("<table style='border-collapse:collapse;width:100%;border:1px solid #ddd;'>")
        h.append("<tr style='background:#ca5010;color:white;'><th style='padding:8px;text-align:left;'>Setting</th>")
        h.append("<th style='padding:8px;text-align:left;'>Field</th>")
        h.append("<th style='padding:8px;text-align:left;'>Old</th>")
        h.append("<th style='padding:8px;text-align:left;'>New</th></tr>")
        rows = spark.sql(f"""
            SELECT setting_name, field_changed, old_value, new_value
            FROM {CHANGES_TABLE}
            WHERE change_date = '{today}' AND change_type = 'CHANGED'
            ORDER BY setting_name, field_changed
        """).collect()
        prev_name = ""
        for r in rows:
            d = r.asDict()
            name = d['setting_name'] if d['setting_name'] != prev_name else ''
            prev_name = d['setting_name']
            h.append(f"<tr style='border-bottom:1px solid #eee;'>")
            h.append(f"<td style='padding:6px 8px;font-weight:bold;'>{name}</td>")
            h.append(f"<td style='padding:6px 8px;'>{d['field_changed']}</td>")
            h.append(f"<td style='padding:6px 8px;color:#d83b01;'>{d['old_value']}</td>")
            h.append(f"<td style='padding:6px 8px;color:#107c10;'>{d['new_value']}</td></tr>")
        h.append("</table><br/>")

    h.append("<hr/><p style='font-size:11px;color:#888;'>Sent by Fabric Tenant Settings Change Tracker pipeline.</p>")
    h.append("</body></html>")
    return "".join(h)


total = new_count + removed_count + changed_count
has_changes = yesterday_count > 0 and total > 0

exit_payload = _json.dumps({
    "has_changes": has_changes,
    "total_changes": total,
    "new_count": new_count,
    "removed_count": removed_count,
    "changed_count": changed_count,
    "subject": f"[Fabric Admin] {total} Tenant Setting Change(s) on {today}" if has_changes
               else f"[Fabric Admin] No Tenant Setting Changes on {today}",
    "body_html": build_email_html() if has_changes else ""
})

print(f"has_changes={has_changes}, total={total}")
mssparkutils.notebook.exit(exit_payload)
